# SecureFL Unique Contribution Showcase

This notebook demonstrates the **new unique components** added to the project:

1. Adaptive trust-weighted robust aggregation
2. Attack simulation + anomaly filtering
3. Non-IID label skew + temporal drift simulation
4. Optional DP-noise perturbation
5. Round-wise audit hash chain


In [1]:
import pandas as pd
from dataclasses import replace

from core.fl_simulation import FLConfig, run_federated_simulation
from security.attack import AttackConfig


In [2]:
fl_cfg = FLConfig(
    num_clients=8,
    num_rounds=8,
    aggregation_method='trust_weighted',
    partition_mode='label_skew',
    enable_drift=True,
    use_dp=True,
    dp_noise_std=0.01,
    encryption_backend='simulated',
)
atk_cfg = AttackConfig(enabled=True, attack_type='scaling', malicious_fraction=0.25)

result_unique = run_federated_simulation(fl_cfg, atk_cfg)
result_unique.summary


{'selected_strategy': 'trust_weighted',
 'strategy_note': '',
 'mean_detection_precision': 1.0,
 'mean_detection_recall': 1.0,
 'final_mean_trust_benign': 0.6259188236986558,
 'final_mean_trust_malicious': 0.1,
 'audit_enabled': True,
 'partition_mode': 'label_skew',
 'dp_enabled': True,
 'drift_enabled': True}

In [3]:
baseline_cfg = replace(fl_cfg, aggregation_method='fedavg')
result_baseline = run_federated_simulation(baseline_cfg, atk_cfg)

comparison = {
    'unique_final_acc': result_unique.final_accuracy_selected_strategy,
    'baseline_final_acc': result_baseline.final_accuracy_selected_strategy,
    'delta': result_unique.final_accuracy_selected_strategy - result_baseline.final_accuracy_selected_strategy,
    'unique_precision': result_unique.summary['mean_detection_precision'],
    'unique_recall': result_unique.summary['mean_detection_recall'],
    'audit_chain_valid': result_unique.audit_chain_valid,
}
comparison


{'unique_final_acc': 0.9385964912280702,
 'baseline_final_acc': 0.9385964912280702,
 'delta': 0.0,
 'unique_precision': 1.0,
 'unique_recall': 1.0,
 'audit_chain_valid': True}

In [4]:
rounds_df = pd.DataFrame(result_unique.rounds)
rounds_df[['round_id','accuracy_with_attack','accuracy_after_filtering','accuracy_selected_strategy','detected_clients','detection_precision','detection_recall']].head(10)


,round_id,accuracy_with_attack,accuracy_after_filtering,accuracy_selected_strategy,detected_clients,detection_precision,detection_recall
0,1,0.912281,0.894737,0.894737,"[0, 6]",1.0,1.0
1,2,0.921053,0.903509,0.903509,"[0, 6]",1.0,1.0
2,3,0.921053,0.912281,0.912281,"[0, 6]",1.0,1.0
3,4,0.921053,0.929825,0.929825,"[0, 6]",1.0,1.0
4,5,0.921053,0.938596,0.938596,"[0, 6]",1.0,1.0
5,6,0.929825,0.938596,0.938596,"[0, 6]",1.0,1.0
6,7,0.929825,0.938596,0.938596,"[0, 6]",1.0,1.0
7,8,0.929825,0.938596,0.938596,"[0, 6]",1.0,1.0


## Reviewer Pitch

We moved beyond a vanilla FL demo by introducing an adaptive trust-aware robust aggregation pipeline,
with configurable attack models, heterogeneity/drift stress testing, optional DP perturbation, and an
audit chain for round integrity verification.
